<a href="https://colab.research.google.com/github/sanketsahoo40/Diseases_Prediction/blob/main/Diseases_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
!pip install -q xgboost

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [47]:
df = pd.read_csv("/content/synthetic_multi_disease_20000.csv")

print("Dataset shape:", df.shape)
print(df["disease"].value_counts())

Dataset shape: (20000, 35)
disease
Typhoid         5000
Dengue          5000
Malaria         5000
Tuberculosis    5000
Name: count, dtype: int64


In [48]:
df = df.drop(columns=["patient_id"], errors="ignore")

X = df.drop("disease", axis=1)
y = df["disease"]

print("Features:")
print(X.columns.tolist())

Features:
['age', 'gender', 'fever', 'high_fever', 'prolonged_fever', 'chills', 'sweating', 'headache', 'severe_headache', 'body_pain', 'muscle_pain', 'joint_pain', 'fatigue', 'weakness', 'nausea', 'vomiting', 'diarrhea', 'constipation', 'abdominal_pain', 'cough', 'persistent_cough', 'chest_pain', 'shortness_of_breath', 'rash', 'night_sweats', 'weight_loss', 'loss_of_appetite', 'dizziness', 'confusion', 'bleeding', 'pain_behind_eyes', 'coughing_blood', 'symptom_duration_days']


In [49]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Disease mapping:")

for i, disease in enumerate(label_encoder.classes_):
    print(i, "=", disease)

Disease mapping:
0 = Dengue
1 = Malaria
2 = Tuberculosis
3 = Typhoid


In [50]:
categorical_features = ["gender"]

numerical_features = [
    col for col in X.columns
    if col != "gender"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "gender",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 16000
Testing samples: 4000


In [52]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [53]:
model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist"
)

model.fit(
    X_train_processed,
    y_train
)

print("Model training completed!")

Model training completed!


In [54]:
y_pred = model.predict(X_test_processed)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 87.83%


In [55]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

              precision    recall  f1-score   support

      Dengue       0.87      0.87      0.87      1000
     Malaria       0.80      0.77      0.78      1000
Tuberculosis       0.99      1.00      0.99      1000
     Typhoid       0.85      0.88      0.86      1000

    accuracy                           0.88      4000
   macro avg       0.88      0.88      0.88      4000
weighted avg       0.88      0.88      0.88      4000



In [56]:
joblib.dump(model, "xgboost_disease_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Models saved successfully!")

Models saved successfully!


In [57]:
patient = pd.DataFrame([{
    "age": 25,
    "gender": "Male",
    "fever": 1,
    "high_fever": 1,
    "prolonged_fever": 1,
    "chills": 1,
    "sweating": 1,
    "headache": 0,
    "severe_headache": 0,
    "body_pain": 0,
    "muscle_pain": 0,
    "joint_pain": 0,
    "fatigue": 1,
    "weakness": 1,
    "nausea": 0,
    "vomiting": 0,
    "diarrhea": 1,
    "constipation": 1,
    "abdominal_pain": 1,
    "cough": 0,
    "persistent_cough": 0,
    "chest_pain": 0,
    "shortness_of_breath": 0,
    "rash": 0,
    "night_sweats": 0,
    "weight_loss": 0,
    "loss_of_appetite": 0,
    "dizziness": 0,
    "confusion": 0,
    "bleeding": 0,
    "pain_behind_eyes": 0,
    "coughing_blood": 0,
    "symptom_duration_days": 2
}])

In [58]:
patient_processed = preprocessor.transform(patient)

prediction = model.predict(patient_processed)

predicted_disease = label_encoder.inverse_transform(prediction)[0]

print("Predicted Disease:", predicted_disease)

Predicted Disease: Typhoid


In [59]:
probabilities = model.predict_proba(patient_processed)[0]

for disease, probability in zip(
    label_encoder.classes_,
    probabilities
):
    print(f"{disease}: {probability * 100:.2f}%")

Dengue: 0.21%
Malaria: 8.76%
Tuberculosis: 0.01%
Typhoid: 91.02%
